In [1]:
from data.data_pipeline import window_purity_stats, load_pamap2, filter_loco_pam, remove_zero_label_rows, incomplete_labeled_rows, interpolation_pam, acc_data_scaling_pam, mag_data_norm_pam, mag_data_rotation_pam, downsample_pam, sliding_window
from sklearn.preprocessing import StandardScaler, LabelEncoder
from torch.utils.data import DataLoader
from sklearn.model_selection import StratifiedGroupKFold
from pathlib import Path
import glob
import pandas as pd
import numpy as np
import torch
import sys
import gc
from MambaClassificationModel import MambaClassificationModel, HARMambaConfig
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from data.preprocessing import fit_labelencoder
from test_mamba import test_model, save_json 
from mamba_ssm.modules.mamba2 import Mamba2
import mamba_ssm.modules.mamba2 as mm
from datetime import datetime
import torch.nn.functional as F
import mamba_ssm
import torch.nn as nn
from tqdm import tqdm
from utils import set_seed
import argparse
import json
import os
from torch.profiler import profile, ProfilerActivity
from data.preprocessing import fit_labelencoder, Dataset_HAR
import time 
try:
    from mamba_ssm.ops.triton.layer_norm import RMSNorm, layer_norm_fn, rms_norm_fn
except ImportError:
    RMSNorm, layer_norm_fn, rms_norm_fn = None, None, None

print(f"Mamba version: {mamba_ssm.__version__}")
print(f"Path: {mm.__file__}")

ImportError: cannot import name 'window_purity_stats' from 'data.data_pipeline' (/home/kmercad/mamba_har/SUPERVISED MAMBA/Mamba_Baseline_PAM/data/data_pipeline.py)

In [ ]:
!hostname
import sys, os, torch
print("python:", sys.executable)
print("cuda visible devices:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())
#!nvidia-smi

In [2]:
def data_split_PAM(fold_id: int = 1) -> tuple[list, list, list]:
    '''
    Predefined subject split for PAMAP2 dataset.
    For a given fold_id, the corresponding subjects are held out as test.
    
    fold_id 1 → test: 101, 102 | val: 103, 104 | train: 105, 106, 107, 108
    fold_id 2 → test: 107, 108 | val: 101, 102 | train: 103, 104, 105, 106
    fold_id 3 → test: 105, 106 | val: 107, 108 | train: 101, 102, 103, 104
    fold_id 4 → test: 103, 104 | val: 105, 106 | train: 107, 108, 101, 102
    '''
    splits_pam = {
        1: {"test": ["101", "102"], "val": ["103", "104"], "train": ["105", "106", "107", "108"]},
        2: {"test": ["107", "108"], "val": ["101", "102"], "train": ["103", "104", "105", "106"]},
        3: {"test": ["105", "106"], "val": ["107", "108"], "train": ["101", "102", "103", "104"]},
        4: {"test": ["103", "104"], "val": ["105", "106"], "train": ["107", "108", "101", "102"]}
    }

    split = splits_pam[fold_id]
    base_path = '../DATASETS/PAMAP2_Dataset/Protocol/'

    training_files = [base_path + 'subject' + subject + '.dat' for subject in split["train"]]
    validation_files = [base_path + 'subject' + subject + '.dat' for subject in split["val"]]
    test_files = [base_path + 'subject' + subject + '.dat' for subject in split["test"]]

    return training_files, validation_files, test_files

In [4]:
training_files, validation_files, test_files = data_split_PAM(4)

In [5]:
def print_post_standardize_stats(df: pd.DataFrame, name: str, acc_gyro_cols: list[int]):
    x = df.iloc[:, acc_gyro_cols].to_numpy(dtype=np.float64)
    ch_mean = x.mean(axis=0)
    ch_std = x.std(axis=0)

    print(f"\n{'-'*30} {name} ACC/GYRO after StandardScaler {'-'*30}")
    print(f"global mean: {x.mean():.6f}")
    print(f"global std : {x.std():.6f}")
    print(f"max |channel mean| : {np.abs(ch_mean).max():.6f}")
    print(f"median |channel mean| : {np.median(np.abs(ch_mean)):.6f}")
    print(f"channel std range : [{ch_std.min():.6f}, {ch_std.max():.6f}]")

def print_mag_rotation_stats(Xw: np.ndarray, name: str):
    mag_cols = [axis + offset for offset in range(6, 27, 9) for axis in range(3)]
    M = Xw[:, :, mag_cols]                     # [W, L, 15]
    per_window_mean = M.mean(axis=1)           # [W, 15]
    flat = M.reshape(-1, M.shape[-1])          # [W*L, 15]

    print(f"\n{'-'*30} {name} MAG after window demeaning {'-'*30}")
    print(f"max |window mean|  : {np.abs(per_window_mean).max():.6e}")
    print(f"mean |window mean| : {np.abs(per_window_mean).mean():.6e}")
    print(f"channel global mean range : [{flat.mean(axis=0).min():.6e}, {flat.mean(axis=0).max():.6e}]")
    print(f"channel std range         : [{flat.std(axis=0).min():.6f}, {flat.std(axis=0).max():.6f}]")

def print_window_label_distribution(yw: np.ndarray, name: str):
    vals, cnts = np.unique(yw, return_counts=True)
    total = len(yw)
    print(f"\n{name} window label distribution")
    for v, c in zip(vals, cnts):
        print(f"label {v}: {c} ({c/total:.4f})")

def print_top_shifted_channels(df, cols, name, top_k=10):
    X = df.iloc[:, cols].to_numpy(dtype=np.float64)
    ch_mean = X.mean(axis=0)
    ch_std = X.std(axis=0)

    print(f"\n{name} top channels by |mean|:")
    for idx in np.argsort(np.abs(ch_mean))[::-1][:top_k]:
        print(f"channel {cols[idx]} | mean={ch_mean[idx]:.6f} | std={ch_std[idx]:.6f}")

    print(f"\n{name} top channels by std deviation from 1:")
    for idx in np.argsort(np.abs(ch_std - 1))[::-1][:top_k]:
        print(f"channel {cols[idx]} | mean={ch_mean[idx]:.6f} | std={ch_std[idx]:.6f}")

def print_outlier_rates(df, cols, name, thr=5.0):
    X = df.iloc[:, cols].to_numpy(dtype=np.float64)
    rates = (np.abs(X) > thr).mean(axis=0)
    print(f"\n{name} outlier rates |z|>{thr}:")
    for idx in np.argsort(rates)[::-1][:10]:
        print(f"channel {cols[idx]} | rate={rates[idx]:.6f}")

def clipping_acc_gyro_z(dataframe: pd.DataFrame, acc_gyro_cols: list[int], zmax: float = 6.0) -> pd.DataFrame:
    '''
    Clip standarized ACC/GYRO to  reduced the outliers [-z, z]
    '''
    dataframe_ = dataframe.copy()
    x = dataframe_.iloc[:, acc_gyro_cols].to_numpy(dtype=np.float32)
    x = np.clip(x, -zmax, zmax)
    dataframe_.iloc[:, acc_gyro_cols] = x
    return dataframe_
        
def load_PAM_loco_data(training_files, validation_files, test_files, verbose = False) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    '''
    Load PAMAP2 dataset and process all the data for the model
    '''
    training_data = load_pamap2(training_files, add_group_id = True)
    validation_data = load_pamap2(validation_files, add_group_id = True)
    test_data = load_pamap2(test_files, add_group_id = True)
    
    # ----- Selection of the Columns -----
    training_data_selected = filter_loco_pam(training_data)
    validation_data_selected = filter_loco_pam(validation_data)
    test_data_selected = filter_loco_pam(test_data)
    gc.collect()

    # ----- Clean label 0 ------ #MOD: REMOVE ALL ROWS WITH LABEL 0. 
    training_data_nan = remove_zero_label_rows(training_data_selected)
    validation_data_nan = remove_zero_label_rows(validation_data_selected)
    test_data_nan = remove_zero_label_rows(test_data_selected)
    if verbose:
        print(f"{'-'*90}")
        print(f"Sensor subset training shape: {training_data_nan.shape}\nSensor Validation training shape: {validation_data_nan.shape}\nSensor Test training shape: {test_data_nan.shape}")  
    gc.collect()
    
     # ----- For verification if there are rows with any NaN value  ----- 
    training_nan = incomplete_labeled_rows(training_data_nan)
    validation_nan = incomplete_labeled_rows(validation_data_nan)
    test_nan = incomplete_labeled_rows(test_data_nan)
    if verbose:
        print(f"{'-'*70}")
        print(f"Rows containing NaNs - training: {training_nan}\nRows containing NaNs - validation: {validation_nan}\nRows containing NaNs - test: {test_nan}")  
        
    # ----- Reset index ----- 
    training_data_nan = training_data_nan.reset_index(drop=True)
    validation_data_nan = validation_data_nan.reset_index(drop=True)
    test_data_nan = test_data_nan.reset_index(drop=True) 
    if verbose:
        print(f"{'-'*90}")
        print(f"Reindexed training shape: {training_data_nan.shape}\nReindexed validation shape: {validation_data_nan.shape}\nReindexed test shape: {test_data_nan.shape}")
   
    # ----- Interpolation -----
    training_data_cleaned = interpolation_pam(training_data_nan, max_gap=30)
    validation_data_cleaned = interpolation_pam(validation_data_nan, max_gap=30)
    test_data_cleaned = interpolation_pam(test_data_nan, max_gap=30)      
    if verbose:
        print(f"{'-'*90}")
        print(f"After Interpolation. Training shape: {training_data_cleaned.shape}\nAfter Interpolation. Validation shape: {validation_data_cleaned.shape}\nAfter Interpolation. Test shape: {test_data_cleaned.shape}")   
    gc.collect() 
    
# --------------------------------------------------
# 1) physics-based scaling
# --------------------------------------------------
    if verbose: 
        print("\n")
        print("Descriptive statistics for ACC channels (DEVICE_1) - training set")
        print(training_data_cleaned.iloc[:, 0:3].describe())
        print("\nDescriptive statistics for GYRO channels (DEVICE_1) - training set")
        print(training_data_cleaned.iloc[:, 3:6].describe())
        print("\nDescriptive statistics for MAG channels (DEVICE_1) - training set")
        print(training_data_cleaned.iloc[:, 6:9].describe())
        print(f"{'-'*90}")
    
    # SCALING ACC
    training_data_scaled = acc_data_scaling_pam(training_data_cleaned)
    validation_data_scaled = acc_data_scaling_pam(validation_data_cleaned)
    test_data_scaled = acc_data_scaling_pam(test_data_cleaned)
    
    # NORM MAG               
    training_data_scaled = mag_data_norm_pam(training_data_scaled)
    validation_data_scaled = mag_data_norm_pam(validation_data_scaled)
    test_data_scaled = mag_data_norm_pam(test_data_scaled)
    
    if verbose:
        print("\n")
        print("Descriptive statistics for ACC channels (DEVICE_1) - training set")
        print(training_data_scaled.iloc[:, 0:3].describe())
        print("\nDescriptive statistics for scaled GYRO channels (DEVICE_1) - training set")
        print(training_data_scaled.iloc[:, 3:6].describe())
        print("\nDescriptive statistics for MAG channels (DEVICE_1) - training set")
        print(training_data_scaled.iloc[:, 6:9].describe())

# --------------------------------------------------
# 2) columns to standardize ACC AND GYRO
# --------------------------------------------------    
    scaler = StandardScaler()
    acc_gyro_cols = [axis + offset for offset in range(0, 27, 9) for axis in range(6)]
    
    training_data_scaled.iloc[:, acc_gyro_cols] = scaler.fit_transform(training_data_scaled.iloc[:, acc_gyro_cols].values)
    validation_data_scaled.iloc[:, acc_gyro_cols] = scaler.transform(validation_data_scaled.iloc[:, acc_gyro_cols].values)
    test_data_scaled.iloc[:, acc_gyro_cols] = scaler.transform(test_data_scaled.iloc[:, acc_gyro_cols].values)
    ############################################
    # Stratify
    combined_eval = pd.concat([validation_data_scaled, test_data_scaled], axis = 0, ignore_index = True) 
    y = combined_eval.iloc[:, -2].values       # Labels
    groups = combined_eval["group_id"].values  # File id Name
        
    sgkf = StratifiedGroupKFold(n_splits = 2, shuffle = True, random_state = 42)
    val_idx, test_idx = next(sgkf.split(combined_eval, y, groups))
    print(f"{'-'*20}STRATIY VAL/TEST{'-'*20}")
    print("Validation GROUPS")
    print(combined_eval.iloc[val_idx]["group_id"].value_counts().sort_index())
    print("Test GROUPS")
    print(combined_eval.iloc[test_idx]["group_id"].value_counts().sort_index(), "\n")
    
    
    new_validation_data_scaled = combined_eval.iloc[val_idx].copy()
    new_test_data_scaled = combined_eval.iloc[test_idx].copy()
    print(f"\n Validation Split Label Proportion")
    print(new_validation_data_scaled.iloc[:, -2].value_counts(normalize=True).sort_index())
    print("Test Split Label Proportion")
    print(new_test_data_scaled.iloc[:, -2].value_counts(normalize=True).sort_index())

    if verbose:
        print_post_standardize_stats(training_data_scaled, "TRAIN", acc_gyro_cols)
        print_post_standardize_stats(new_validation_data_scaled, "VAL", acc_gyro_cols)
        print_post_standardize_stats(new_test_data_scaled, "TEST", acc_gyro_cols)

        print_top_shifted_channels(training_data_scaled, acc_gyro_cols, "TRAIN")
        print_top_shifted_channels(new_validation_data_scaled, acc_gyro_cols, "VAL")
        print_top_shifted_channels(new_test_data_scaled, acc_gyro_cols, "TEST")

        print_outlier_rates(training_data_scaled, acc_gyro_cols, "TRAIN")
        print_outlier_rates(new_validation_data_scaled, acc_gyro_cols, "VAL")
        print_outlier_rates(new_test_data_scaled, acc_gyro_cols, "TEST")

    #training_data_scaled = clipping_acc_gyro_z(training_data_scaled, acc_gyro_cols)
    #new_validation_data_scaled = clipping_acc_gyro_z(new_validation_data_scaled, acc_gyro_cols)
    #new_test_data_scaled = clipping_acc_gyro_z(new_test_data_scaled, acc_gyro_cols)

    #if verbose:
        #print_outlier_rates(training_data_scaled, acc_gyro_cols, "TRAIN after clip")
        #print_outlier_rates(new_validation_data_scaled, acc_gyro_cols, "VAL after clip")
        #print_outlier_rates(new_test_data_scaled, acc_gyro_cols, "TEST after clip")
    ############################################
    train_ds = downsample_pam(training_data_scaled, group_id="group_id", factor=3)
    val_ds   = downsample_pam(new_validation_data_scaled, group_id="group_id", factor=3)
    test_ds  = downsample_pam(new_test_data_scaled, group_id="group_id", factor=3)

    # ----- Sliding Windows -----
    #X_windows, y_windows = create_pamap_windows_from_split(training_data_scaled, group_id = "group_id", window_size = 500, stride = 250, target_len = 150, resample_method = "median")
    #X_validation_windows, y_validation_windows = create_pamap_windows_from_split(new_validation_data_scaled, group_id = "group_id", window_size = 500, stride = 250, target_len = 150, resample_method = "median")
    #X_test_windows, y_test_windows = create_pamap_windows_from_split(new_test_data_scaled, group_id = "group_id", window_size = 500, stride = 250, target_len = 150, resample_method = "median")
    # TRAIN
    X_all, y_all = [], []
    for _, gdf in train_ds.groupby("group_id", sort=False):
        gdf = gdf.drop(columns=["group_id"]).reset_index(drop=True)
        X_features = gdf.iloc[:, :-1]
        y_labels = gdf.iloc[:, -1]
        window_purity_stats(y_labels, window_size=171, stride=38, name="TRAIN")
        Xw, yw = sliding_window(X_features, y_labels, window_size=171, stride=38)
        X_all.append(Xw)
        y_all.append(yw)
    
    X_windows = np.concatenate(X_all, axis=0).astype(np.float32)
    y_windows = np.concatenate(y_all, axis=0).astype(np.int64)
    
    
    # VALIDATION
    X_val_all, y_val_all = [], []
    for _, gdf in val_ds.groupby("group_id", sort=False):
        gdf = gdf.drop(columns=["group_id"]).reset_index(drop=True)
        X_val_features = gdf.iloc[:, :-1]
        y_val_labels = gdf.iloc[:, -1]
        window_purity_stats(y_labels, window_size=171, stride=38, name="VAL")
        Xw_val, yw_val = sliding_window(X_val_features, y_val_labels, window_size=171, stride=38)
        X_val_all.append(Xw_val)
        y_val_all.append(yw_val)
    
    X_validation_windows = np.concatenate(X_val_all, axis=0).astype(np.float32)
    y_validation_windows = np.concatenate(y_val_all, axis=0).astype(np.int64)
    
    
    # TEST
    X_test_all, y_test_all = [], []
    for _, gdf in test_ds.groupby("group_id", sort=False):
        gdf = gdf.drop(columns=["group_id"]).reset_index(drop=True)
        X_test_features = gdf.iloc[:, :-1]
        y_test_labels = gdf.iloc[:, -1]
        window_purity_stats(y_labels, window_size=171, stride=38, name="TEST")
        Xw_test, yw_test = sliding_window(X_test_features, y_test_labels, window_size=171, stride=38)
        X_test_all.append(Xw_test)
        y_test_all.append(yw_test)
    
    X_test_windows = np.concatenate(X_test_all, axis=0).astype(np.float32)
    y_test_windows = np.concatenate(y_test_all, axis=0).astype(np.int64)

    #MAG per window demeaning
    X_windows = mag_data_rotation_pam(X_windows)
    X_validation_windows = mag_data_rotation_pam(X_validation_windows)
    X_test_windows = mag_data_rotation_pam(X_test_windows)

    if verbose:
        print_mag_rotation_stats(X_windows, "TRAIN")
        print_mag_rotation_stats(X_validation_windows, "VAL")
        print_mag_rotation_stats(X_test_windows, "TEST")
    
        print_window_label_distribution(y_windows, "TRAIN")
        print_window_label_distribution(y_validation_windows, "VAL")
        print_window_label_distribution(y_test_windows, "TEST")
    
    if verbose:
        print(f"{'-'*90}")
        print(f"Training (windows): {X_windows.shape}. Training Labels {y_windows.shape}")
        print(f"Validation (windows): {X_validation_windows.shape}. Validation Labels {y_validation_windows.shape}")
        print(f"Test (windows): {X_test_windows.shape}. Test Labels {y_test_windows.shape}")
    return X_windows, y_windows, X_validation_windows, y_validation_windows, X_test_windows, y_test_windows

    

In [6]:
X_windows, y_windows, X_validation_windows, y_validation_windows, X_test_windows, y_test_windows = load_PAM_loco_data(training_files, validation_files, test_files, verbose = True)

------------------------------------------------------------------------------------------
Sensor subset training shape: (399666, 29)
Sensor Validation training shape: (201186, 29)
Sensor Test training shape: (205551, 29)
----------------------------------------------------------------------
Rows containing NaNs - training: 5278
Rows containing NaNs - validation: 2614
Rows containing NaNs - test: 2883
------------------------------------------------------------------------------------------
Reindexed training shape: (399666, 29)
Reindexed validation shape: (201186, 29)
Reindexed test shape: (205551, 29)
------------------------------------------------------------------------------------------
After Interpolation. Training shape: (399666, 29)
After Interpolation. Validation shape: (201024, 29)
After Interpolation. Test shape: (205551, 29)


Descriptive statistics for ACC channels (DEVICE_1) - training set
                   0              1              2
count  399666.000000  399666.00

In [7]:
def make_loaders_PAM(X_windows, y_windows, X_validation_windows, y_validation_windows, X_test_windows, y_test_windows, generator, verbose=False) -> tuple[DataLoader, DataLoader, DataLoader, LabelEncoder]:
    '''
    Creates DataLoaders for training, validation and test sets.
    Called once per seed — generator ensures reproducible shuffling.

    Args:
    X_windows, y_windows: training windows and labels
    X_validation_windows, y_validation_windows: validation windows and labels
    X_test_windows, y_test_windows: test windows and labels
    generator: seeded for reproducibility
    verbose: print class counts and batch shapes
    '''
    # ----- LabelEncoder, Transform, DataLoader -----
    label_encoder = fit_labelencoder(X_windows, y_windows)
    training_dataset = Dataset_HAR(X_windows, y_windows, label_encoder=label_encoder)
    validation_dataset = Dataset_HAR(X_validation_windows, y_validation_windows, label_encoder=label_encoder)
    test_dataset = Dataset_HAR(X_test_windows, y_test_windows, label_encoder=label_encoder)

    train_loader = DataLoader(training_dataset, batch_size = 32, shuffle = True, generator = generator, num_workers = 0) #pin_memory = True, persistent_workers = True)
    val_loader = DataLoader(validation_dataset, batch_size = 32, shuffle = False, num_workers = 0)# pin_memory = True, persistent_workers = True)
    test_loader = DataLoader(test_dataset, batch_size = 32, shuffle = False, num_workers = 0)# pin_memory = True, persistent_workers = True)

    # ----- Samples per label checking and batch size -----
    if verbose:
        label_to_name = {0: "LIE", 1: "SIT", 2: "STAND", 3: "WALK"}
        class_counts = {name: 0 for name in label_to_name.values()}
        for _, y_batch in train_loader:
            for label in y_batch:
                label_idx = label.item()
                class_name = label_to_name[label_idx]
                class_counts[class_name] += 1

        print(f"{'-'*90}")
        print("Training set class distribution:--")
        print(class_counts)
        print(f"{'-'*90}")
    return train_loader, val_loader, test_loader, label_encoder

In [8]:
@torch.no_grad()
def validate_model(model, val_loader, device, criterion):
    '''
    Validation: avg loss per window and accuracy per window.
    '''
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    all_preds = []
    all_labels = []
    
    for batch_idx, (x_batch, y_batch) in enumerate(val_loader):
        x_batch, y_batch = x_batch.to(device, non_blocking = True), y_batch.to(device, non_blocking = True)
        logits_ = model(x_batch)                                               # forward Pass
        loss = criterion(logits_, y_batch)                                     # computes mean batch loss

        bsize = y_batch.size(0)
        total_loss += loss.item() * bsize                                      # Total loss contribution of this batch

        predictions = logits_.argmax(dim = -1)                                 # gets the predicted class for each window
        total_correct += (predictions == y_batch).sum().item()                 # counts correct predictions
        total_samples += bsize

        all_preds.extend(predictions.cpu().numpy())
        all_labels.extend(y_batch.cpu().numpy())

        if batch_idx == 0:  # just first batch, so it doesn't spam too much
            true_ids, true_counts = torch.unique(y_batch, return_counts=True)
            pred_ids, pred_counts = torch.unique(predictions, return_counts=True)
    report = classification_report(all_labels, all_preds, target_names = ["LIE", "SIT", "STAND", "WALK"])
    return total_loss / total_samples, total_correct / total_samples, report

In [9]:
#  ----------------------------------------------------------------------------------------------------------------------
#  ---------------------------------------------------- LOSO TRAINING ---------------------------------------------------
for seed in [42, 58, 7, 128, 92]: # [42, 58, 7, 128, 92]
    g = set_seed(seed)
    train_loader, val_loader, test_loader, label_encoder = make_loaders_PAM(X_windows, y_windows, X_validation_windows, y_validation_windows, X_test_windows, y_test_windows, generator = g, verbose = True)
    
    #  ----------- TRAINING SETUP -----------
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    num_classes = int(len(label_encoder.classes_))
    pam_config = HARMambaConfig(num_sensor_features=X_windows.shape[2])
    model = MambaClassificationModel(config = pam_config, num_classes = num_classes).to(device, non_blocking = True) 
    
    # ----------------------
    num_epochs = 25
    lr = 0.00001
    patience = 4
    #WCE
    y_train_encoded = label_encoder.transform(np.asarray(y_windows))
    counts = np.bincount(y_train_encoded, minlength=num_classes)
    weights = 1.0 / np.sqrt(counts)
    weights = torch.tensor(weights, dtype=torch.float32, device=device)
    weights = weights / weights.sum() * len(weights)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr = lr, weight_decay = 5e-4)

    #  ----------- TRAINING -----------
    model_name = f"models_pt/model_OPP_fold{1}_seed{seed}.pt"
    epoch_history = []
    best_val_loss = float("inf")
    best_val_acc = 0.0
    best_epoch = None
    best_state = None
    bad_epochs = 0
    with open(f"logs/training_OPP_fold{4}.txt", "a") as log_file:
        log_file.write(f"\nTRAINING STARTING AT: {datetime.now()}\n")
        log_file.write(f"Model: {model_name} | SEED: {seed}\n")
        log_file.flush()
        for epoch in range(num_epochs):
            epoch_start = time.time() # START EPOCH TIME
            model.train()
            total_loss = 0.0
            total_correct = 0
            total_samples = 0
    
            loop = tqdm(train_loader, desc= f"Epoch {epoch+1}/{num_epochs}")
            for batch_idx, (x_batch, y_batch) in enumerate(loop):
                x_batch, y_batch = x_batch.to(device, non_blocking = True), y_batch.to(device, non_blocking = True)    
                optimizer.zero_grad()                                             # clear previous gradients
                logits_ = model(x_batch)                                          # forward pass
                loss = criterion(logits_, y_batch)                                # computes mean batch loss
                loss.backward()                                                   # backward pass: compute gradients
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # prevent exploding gradients
                optimizer.step() 
   
                bsize = y_batch.size(0)
                total_loss += loss.item() * bsize                                 # Total loss contribution of this batch
                
                predictions = logits_.argmax(dim = -1)                            # gets the predicted class for each window: picks the highest score in the last dimension (one highest score per window among the 4 classes)
                total_correct += (predictions == y_batch).sum().item()            # number of correct predictions
                total_samples += bsize
                loop.set_postfix(loss= f"{total_loss/total_samples:.4f}", acc=f"{total_correct/total_samples:.4f}")
    
            train_loss, train_acc = total_loss / total_samples, total_correct / total_samples
            val_loss, val_acc, report = validate_model(model, val_loader, device, criterion)
            if device.type == "cuda":
                torch.cuda.synchronize()
            epoch_time = time.time() - epoch_start # END EPOCH TIME
            epoch_history.append({
                "epoch": epoch + 1,
                "tr_loss": float(train_loss),
                "tr_acc": float(train_acc),
                "val_loss": float(val_loss),
                "val_acc": float(val_acc)                
            })
            print(f"\nEpoch: {epoch+1}/{num_epochs} | tr_Loss: {train_loss:.4f} | tr_acc: {train_acc:.4f} | val_loss: {val_loss:.4f} | val_acc: {val_acc:.4f} | epoch_time: {epoch_time:.2f}s")   
            log_file.write(f"Epoch: {epoch+1}/{num_epochs} | tr_loss: {train_loss:.4f} | tr_acc: {train_acc:.4f} | val_loss: {val_loss:.4f} | val_acc: {val_acc:.4f} | epoch_time: {epoch_time:.2f}s\n")
            log_file.flush() 

            if val_loss < best_val_loss - 1e-12:
                best_val_loss = float(val_loss)
                best_val_acc = float(val_acc)
                best_epoch = epoch + 1                   
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                bad_epochs = 0
            else:
                bad_epochs += 1
                if bad_epochs >= patience:
                    print(f"\nEarly stopping at epoch {epoch + 1} | Best Validation Loss: {best_val_loss:.4f}")
                    break
                    
        # Load Best Model !            
        if best_state is not None:
            model.load_state_dict(best_state)
            torch.save(best_state, model_name)
            # Run val on best model to generate report
            _, _, report = validate_model(model, val_loader, device, criterion) 
            print(report)
            log_file.write(f"\nBest Model Validation Report: \n{str(report)}")        
        log_file.write(f"TRAINING ENDING AT: {datetime.now()}\n")                           
        # ----------------------        
        acc, report, f1, conf_matrix = test_model(model, test_loader, device)
        seed_result = {
            "seed": int(seed),
            "fold": int(1),
            "history": epoch_history,
            "summary": {
                "best_epoch": best_epoch,
                "best_val_loss": float(best_val_loss),
                "best_val_acc": float(best_val_acc),
                "test_accuracy": float(acc),
                "test_report": report,
                "test_f1": float(f1),
                "test_conf_matrix": conf_matrix.tolist()               
            }
        }
        save_json("OPP", "1", seed, seed_result)
        print(f"{'-'*90}")
        print(f"Test Results:\n Accuracy: {acc}\n Report:\n {report}\n F1: {f1}\n Confusion Matrix:\n {conf_matrix}")
        log_file.write(f"\nTest Results:\n Accuracy: {acc}\n Report:\n {report}\n F1: {f1}\n Confusion Matrix:\n {conf_matrix}")
#  ----------------------------------------------------------------------------------------------------------------------

------------------------------------------------------------------------------------------
Training set class distribution:--
{'LIE': 874, 'SIT': 711, 'STAND': 861, 'WALK': 1043}
------------------------------------------------------------------------------------------


Epoch 1/25: 100%|██████████| 110/110 [00:28<00:00,  3.91it/s, acc=0.8770, loss=0.7091]



Epoch: 1/25 | tr_Loss: 0.7091 | tr_acc: 0.8770 | val_loss: 0.5732 | val_acc: 0.8886 | epoch_time: 28.84s


Epoch 2/25: 100%|██████████| 110/110 [00:05<00:00, 21.97it/s, acc=0.9604, loss=0.2444]



Epoch: 2/25 | tr_Loss: 0.2444 | tr_acc: 0.9604 | val_loss: 0.4075 | val_acc: 0.9000 | epoch_time: 5.70s


...


Epoch: 11/25 | tr_Loss: 0.0222 | tr_acc: 0.9971 | val_loss: 0.3184 | val_acc: 0.9131 | epoch_time: 5.52s


Epoch 12/25: 100%|██████████| 110/110 [00:04<00:00, 22.82it/s, acc=0.9968, loss=0.0193]



Epoch: 12/25 | tr_Loss: 0.0193 | tr_acc: 0.9968 | val_loss: 0.2992 | val_acc: 0.9273 | epoch_time: 5.53s


Epoch 13/25: 100%|██████████| 110/110 [00:04<00:00, 22.84it/s, acc=0.9980, loss=0.0155]



Epoch: 13/25 | tr_Loss: 0.0155 | tr_acc: 0.9980 | val_loss: 0.2956 | val_acc: 0.9290 | epoch_time: 5.51s

Early stopping at epoch 13 | Best Validation Loss: 0.2736
              precision    recall  f1-score   support

         LIE       0.95      0.97      0.96       403
         SIT       0.85      0.93      0.89       425
       STAND       0.93      0.82      0.87       431
        WALK       0.98      0.98      0.98       501

    accuracy                           0.93      1760
   macro avg       0.93      0.92      0.92      1760
weighted avg       0.93      0.93      0.93      1760

------------------------------------------------------------------------------------------
Test Results:
 Accuracy: 0.8094972067039106
 Report:
 {'LIE': {'precision': 0.9816272965879265, 'recall': 0.9396984924623115, 'f1-score': 0.9602053915275995, 'support': 398.0}, 'SIT': {'precision': 0.7885196374622356, 'recall': 0.5359342915811088, 'f1-score': 0.6381418092909535, 'support': 487.0}, 'STAND': {